# Projet : Système de recommandation d'images

## Groupe

- Mathis LAMBERT
- Quentin BOUCHOT
- Hugo RODRIGUES


## Présentation du projet

Comme indiqué dans [Projet.md](fr/Projet/Projet.md), l'objectif global est de construire un **système de recommandation d'images** capable de proposer des images à des utilisateurs selon leurs préférences.

Le projet mobilise plusieurs étapes : collecte des données, enrichissement des images, analyse, visualisation puis recommandation.

Pour notre étude, nous avons choisi de travailler sur un jeu de données de **fleurs**, car il contient des images riches en couleurs et bien adaptées à une première analyse visuelle avec `KMeans`.

Dataset choisi : [pufanyi/flowers102](https://huggingface.co/datasets/pufanyi/flowers102)


## Tâche 1 : Collecte de données

D'après les consignes du projet, cette première tâche demande de collecter au moins **100 images sous licence libre** ainsi que leurs **métadonnées**.

Dans ce notebook, nous utilisons un échantillon de 200 images du dataset Flowers 102 afin de disposer d'une base simple pour tester ensuite l'extraction des couleurs dominantes.


## Environnement

La cellule suivante permet d'installer la bibliothèque `datasets` si elle n'est pas encore disponible dans l'environnement Jupyter.


In [ ]:
!pip install datasets


## Chargement des données

Nous chargeons ici un échantillon de 200 images pour garder un notebook lisible et raisonnable en temps d'exécution.


In [ ]:
from datasets import load_dataset

ds = load_dataset("pufanyi/flowers102", split="train", streaming=True)

samples = []
for i, sample in enumerate(ds):
    if i == 200:
        break
    samples.append(sample)

len(samples)


## Préparation des images

Nous convertissons les images en RGB puis nous les stockons dans un `DataFrame` pandas.

Cette étape prépare les données pour les traitements de clustering et pour l'ajout futur de nouvelles métadonnées.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# CONSTANTS
IMAGE_MAX_SIZE = 64

df = pd.DataFrame({
    "label": [sample["label"] for sample in samples],
    "image": [sample["image"].convert("RGB") for sample in samples],
})

df.head()

## Méthode de clustering des couleurs

Nous définissons ici deux fonctions simples :

- une fonction qui redimensionne l'image ;
- une fonction qui applique `KMeans` sur les pixels RGB.

Chaque pixel est considéré comme un point dans un espace à 3 dimensions : rouge, vert et bleu.


In [ ]:
def resize_image(image, max_size=IMAGE_MAX_SIZE):
    image = image.copy()
    image.thumbnail((max_size, max_size))
    return image

def cluster_colors(image, n_colors=5):
    resized_image = resize_image(image)

    # Convertir l'image en un tableau de pixels normalisé, on divise par 255
    # pour avoir des valeurs entre 0 et 1.
    pixels = np.array(resized_image, dtype=np.float32) / 255.0

    # Reshaper le tableau pour obtenir une ligne par pixel et trois colonnes RGB.
    pixels_2d = pixels.reshape(-1, 3)

    # Appliquer le clustering KMeans aux pixels de l'image.
    model = KMeans(n_clusters=n_colors, random_state=42, n_init="auto")
    labels = model.fit_predict(pixels_2d)

    # Reconstruire une image segmentée à partir des centres de clusters.
    centers = model.cluster_centers_
    segmented_pixels = centers[labels].reshape(pixels.shape)
    segmented_image = (segmented_pixels * 255).astype(np.uint8)

    # Calculer la fréquence de chaque couleur dominante.
    counts = np.bincount(labels)
    order = np.argsort(counts)[::-1]

    # Trier la palette par ordre décroissant de fréquence.
    palette = centers[order]
    frequencies = counts[order]

    return resized_image, segmented_image, palette, frequencies


## Choix du nombre de clusters

Avant d'appliquer le clustering à tout le dataset, nous cherchons à choisir une valeur cohérente pour `k`, c'est-à-dire le nombre de couleurs dominantes à extraire.

Pour cela, nous utilisons deux indicateurs complémentaires :

- l'**inertie**, utilisée dans la méthode du coude ;
- le **score de silhouette**, qui mesure la qualité de la séparation entre les clusters.


In [ ]:
from sklearn.metrics import silhouette_score

def find_optimal_n_colors(image, n_min=2, n_max=10, random_state=42):
    resized_image = resize_image(image)

    pixels = np.array(resized_image, dtype=np.float32) / 255.0
    pixels_2d = pixels.reshape(-1, 3)

    ks = list(range(n_min, n_max + 1))
    inertias = []
    silhouettes = []

    for k in ks:
        model = KMeans(n_clusters=k, random_state=random_state, n_init="auto")
        labels = model.fit_predict(pixels_2d)

        # Inertie = somme des distances intra-cluster.
        inertias.append(model.inertia_)

        # Score de silhouette = qualité de séparation des clusters.
        silhouettes.append(silhouette_score(pixels_2d, labels))

    best_index = int(np.nanargmax(silhouettes))
    best_k = ks[best_index]

    return best_k, ks, inertias, silhouettes


In [ ]:
def plot_metrics(ks, inertias, silhouettes):
    fig, ax1 = plt.subplots(figsize=(10, 5))

    color = "tab:blue"
    ax1.set_xlabel("Nombre de clusters (k)")
    ax1.set_ylabel("Inertie", color=color)
    ax1.plot(ks, inertias, marker="o", color=color)
    ax1.tick_params(axis="y", labelcolor=color)

    ax2 = ax1.twinx()
    color = "tab:orange"
    ax2.set_ylabel("Score de silhouette", color=color)
    ax2.plot(ks, silhouettes, marker="o", color=color)
    ax2.tick_params(axis="y", labelcolor=color)

    plt.title("Inertie et score de silhouette en fonction de k")
    plt.show()


## Analyse exploratoire sur un échantillon

Nous appliquons cette analyse seulement sur un petit échantillon d'images, car le calcul du score de silhouette est relativement coûteux.

L'objectif est d'observer les tendances, puis de retenir une valeur fixe de `k` pour le traitement de l'ensemble du dataset.


In [ ]:
analysis_results = []

for image in df["image"][:3]:
    best_k, ks, inertias, silhouettes = find_optimal_n_colors(image, n_min=2, n_max=10)
    analysis_results.append((best_k, ks, inertias, silhouettes))

for i, (best_k, ks, inertias, silhouettes) in enumerate(analysis_results):
    print(f"Image {i} - Meilleur k : {best_k}")
    plot_metrics(ks, inertias, silhouettes)


## Valeur retenue pour le clustering

À partir de cette analyse exploratoire, nous retenons dans la suite une valeur fixe de `k = 4`.

Ce choix permet d'obtenir une segmentation lisible des couleurs tout en gardant un temps de calcul raisonnable sur l'ensemble des images.


In [ ]:
OPTIMAL_CLUSTERS = 4

## Application du clustering sur l'ensemble des images

Nous appliquons maintenant la méthode choisie à toutes les images de l'échantillon.

Nous ajoutons ensuite quatre colonnes au `DataFrame` :

- l'image redimensionnée ;
- l'image segmentée ;
- la palette de couleurs dominantes ;
- la fréquence de chaque couleur.


In [ ]:
results = [cluster_colors(image, n_colors=OPTIMAL_CLUSTERS) for image in df["image"]]

df["resized_image"] = [result[0] for result in results]
df["clustered_image"] = [result[1] for result in results]
df["palette"] = [result[2] for result in results]
df["frequencies"] = [result[3] for result in results]

df[["label"]].head()


## Visualisation des résultats

Pour quelques images, nous affichons :

1. l'image originale redimensionnée ;
2. l'image reconstruite après clustering ;
3. la palette des couleurs dominantes.


In [ ]:
def palette_bar(colors, frequencies, width=300, height=50):
    frequencies = frequencies / frequencies.sum()
    bar = np.zeros((height, width, 3), dtype=np.uint8)

    start = 0
    for color, freq in zip(colors, frequencies):
        end = start + int(freq * width)
        bar[:, start:end] = (color * 255).astype(np.uint8)
        start = end

    return bar

IMAGE_TO_SHOW = 30
sample_ids = np.arange(IMAGE_TO_SHOW)
fig, axes = plt.subplots(len(sample_ids), 3, figsize=(12, 3 * len(sample_ids)))

for row, i in enumerate(sample_ids):
    axes[row, 0].imshow(df.loc[i, "resized_image"])
    axes[row, 0].set_title("Image originale")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(df.loc[i, "clustered_image"])
    axes[row, 1].set_title("Image segmentée")
    axes[row, 1].axis("off")

    bar = palette_bar(df.loc[i, "palette"], df.loc[i, "frequencies"])
    axes[row, 2].imshow(bar)
    axes[row, 2].set_title("Palette dominante")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.show()
